# Preload and resize all images once

In [ ]:
import os, cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

IMG_SIZE = (224, 224)
DATA_DIR = "/content/drive/MyDrive/data"

def load_images(folder, label):
    images, labels = [], []
    for subdir in os.listdir(folder):
        path = os.path.join(folder, subdir)
        for file in tqdm(os.listdir(path), desc=f"Loading {subdir}"):
            img_path = os.path.join(path, file)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, IMG_SIZE)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(label)
    return np.array(images), np.array(labels)

real_imgs, real_lbls = load_images(os.path.join(DATA_DIR, "real"), 1)
fake_imgs, fake_lbls = load_images(os.path.join(DATA_DIR, "fake"), 0)

X = np.concatenate([real_imgs, fake_imgs])
y = np.concatenate([real_lbls, fake_lbls])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y)


Loading 2000: 100%|██████████| 145/145 [00:03<00:00, 43.90it/s]


# Create a fast TensorFlow dataset

In [ ]:
import tensorflow as tf

def make_dataset(images, labels, train=True):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32)),
                num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE)
    else:
        ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train)
val_ds = make_dataset(X_val, y_val, train=False)


# Transfer Learning with MobileNetV2 for Binary Image Classification

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(base_model.input, output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Training the Model with Callbacks

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.2, patience=2),
    ModelCheckpoint("best_model.h5", save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.9072 - loss: 0.2362

187/187 ━━━━━━━━━━━━━━━━━━━━ 57s 201ms/step - accuracy: 0.9073 - loss: 0.2357 - val_accuracy: 0.9711 - val_loss: 0.0765 - learning_rate: 0.0010
Epoch 2/20
186/187 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9679 - loss: 0.0763

187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.9679 - loss: 0.0762 - val_accuracy: 0.9825 - val_loss: 0.0536 - learning_rate: 0.0010
Epoch 3/20
186/187 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.9769 - loss: 0.0631

187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - accuracy: 0.9770 - loss: 0.0631 - val_accuracy: 0.9819 - val_loss: 0.0502 - learning_rate: 0.0010
Epoch 4/20
185/187 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9800 - loss: 0.0471

187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.9800 - loss: 0.0471 - val_accuracy: 0.9812 - val_loss: 0.0491 - learning_rate: 0.0010
Epoch 5/20
185/187 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.9852 - loss: 0.0405

...

187/187 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - accuracy: 0.9991 - loss: 0.0086 - val_accuracy: 0.9906 - val_loss: 0.0329 - learning_rate: 4.0000e-05
Epoch 15/20
185/187 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.9981 - loss: 0.0075

187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.9981 - loss: 0.0075 - val_accuracy: 0.9886 - val_loss: 0.0325 - learning_rate: 4.0000e-05
Epoch 16/20
186/187 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.9989 - loss: 0.0077

187/187 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - accuracy: 0.9989 - loss: 0.0077 - val_accuracy: 0.9899 - val_loss: 0.0321 - learning_rate: 4.0000e-05
Epoch 17/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.9993 - loss: 0.0065 - val_accuracy: 0.9899 - val_loss: 0.0321 - learning_rate: 4.0000e-05
Epoch 18/20
186/187 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.9983 - loss: 0.0072

187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.9983 - loss: 0.0072 - val_accuracy: 0.9906 - val_loss: 0.0319 - learning_rate: 4.0000e-05
Epoch 19/20
186/187 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9979 - loss: 0.0088

187/187 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.9979 - loss: 0.0088 - val_accuracy: 0.9893 - val_loss: 0.0319 - learning_rate: 4.0000e-05
Epoch 20/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.9991 - loss: 0.0065 - val_accuracy: 0.9906 - val_loss: 0.0322 - learning_rate: 4.0000e-05
